In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

from spatial_tcr.pl import utils as plot_utils

figure_dir = "figures/revision/figure-2"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import os

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

from spatial_tcr.pl import plot_cells_view
from spatial_tcr.tcr import get_tcr_genes

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/06.2-kidney_tcr_filtered.h5ad"
adata = sc.read_h5ad(path)
adata.obs["cell_id"] = adata.obs.index.str.split("output").str[0]
adata = adata[adata.obs["condition"] == "ANCA"].copy()
adata

## Find potential examples

In [ ]:
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)

In [ ]:
example_cell_ids = [["iofookae-1", "iogbbaeo-1"]]

In [ ]:
for ci in example_cell_ids:
    ad_tmp = adata[adata.obs["cell_id"].isin(ci), tv_genes].copy()
    var_names = np.array(ad_tmp.var_names)
    X = ad_tmp.layers["counts"].toarray().astype(bool)
    genes = [v for row in X for v in var_names[row]]
    print(genes)

In [ ]:
base_dir = "/bonn-epyc/projects/spatialTCR/20240719__094819__human_kidney_7_TCR"

samples = adata.obs["sample"].unique()
sample_to_data_dir = dict(
    zip(samples, [os.path.join(base_dir, f"output-{sample}") for sample in samples])
)
cc_to_sample = adata.obs[["cc", "sample"]].apply(np.array).drop_duplicates().values
cc_to_sample = dict(zip(cc_to_sample[:, 0], cc_to_sample[:, 1]))

cc_to_data_dir = {cc: sample_to_data_dir[sample] for cc, sample in cc_to_sample.items()}

In [ ]:
index = 0

In [ ]:
xlength = 21
# ylength = int(xlength * (2 / 3))
ylength = xlength

In [ ]:
adata.obs["cell_color"] = adata.obs["cell_type_l2"].astype(str)
adata.obs.loc[~(adata.obs["cell_type_l1"] == "T"), "cell_color"] = "other"
adata.obs.loc[adata.obs["cell_id"].isin(example_cell_ids[index]), "cell_color"] = (
    "highlight_"
    + adata.obs.loc[adata.obs["cell_id"].isin(example_cell_ids[index]), "cell_color"]
)
adata.obs["cell_color"].value_counts()

# adata.obs["cell_color"] = adata.obs["cell_id"]

In [ ]:
from spatial_tcr.colors import colors_sub

palette = colors_sub.copy()
palette["highlight_gdT"] = palette["gdT"]
palette["highlight_CD4+"] = palette["CD4+"]
palette_alpha = {
    ct: mcolors.to_rgba(palette[ct], alpha=0.0 if ct == "NA" else 1)
    for ct in palette.keys()
}
for k in adata.obs["cell_color"].unique():
    if k not in palette_alpha:
        palette_alpha[k] = mcolors.to_rgba("lightgray", alpha=0.5)
    if "highlight" not in k:
        palette_alpha[k] = mcolors.to_rgba(palette_alpha[k], alpha=0.5)

plot_utils.plot_palette_dict(palette_alpha)

In [ ]:
[g for g in adata.var_names if "CD3" in g]

In [ ]:
for cell_ids in example_cell_ids[index : index + 1]:
    # get the av bv dv gv genes expressed in the cells
    ad_tmp = adata[adata.obs["cell_id"].isin(cell_ids), tv_genes].copy()
    var_names = np.array(ad_tmp.var_names)
    X = ad_tmp.layers["counts"].toarray().astype(bool)
    genes = [v for row in X for v in var_names[row]]
    genes = {g: [g] for g in genes}
    if "TRBV6" in genes:
        genes["TRBV6"] = ["TRBV6-1", "TRBV6-2", "TRBV6-5", "TRBV6-7"]
    if "TRBV10" in genes:
        genes["TRBV10"] = ["TRBV10-1", "TRBV10-2", "TRBV10-3"]
    if "TRBV12" in genes:
        genes["TRBV12"] = ["TRBV12-3", "TRBV12-4", "TRBV12-5"]
    if "TRAV12" in genes:
        genes["TRAV12"] = ["TRAV12-1", "TRAV12-2", "TRAV12-3"]
    if "TRBV4" in genes:
        genes["TRBV4"] = ["TRBV4-1", "TRBV4-2"]
    if "TRAV19_21" in genes:
        genes["TRAV19_21"] = ["TRAV19", "TRAV21"]
    print(genes)
    genes["CD3D"] = ["CD3D"]
    genes["CD3E"] = ["CD3E"]
    genes["CD3G"] = ["CD3G"]

    sample = adata[adata.obs["cell_id"].isin(cell_ids)].obs["cc"].iloc[0]
    data_dir = cc_to_data_dir[sample]
    ad_zoom, fig, ax, legends = plot_cells_view(
        adata,
        data_dir=data_dir,
        sample=sample,
        cell_ids=cell_ids,
        color_key="cell_color",
        palette=palette_alpha,
        # palette=None,
        xlength=xlength,
        ylength=ylength,
        genes=genes,
        tick_size=5,
        gene_kwargs={"s": 40},
        img_kwargs={"vmax": 4000},
    )
    plt.savefig(
        os.path.join(figure_dir, "gdT_abT_example.pdf"),
        bbox_inches="tight",
        dpi=300,
        transparent=True,
        pad_inches=0,
    )